# MINI CLASSIFIER TO EXPLORE MODEL MODIFICATIONS: TEST

In [1]:
import os
import ast
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchaudio
import torchaudio.functional as AF
import torch.nn.functional as F

import soundfile as sf
import librosa
import timm

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.metrics import multilabel_confusion_matrix

# UTILS

In [2]:
# Dictionary
path_csv = os.path.expanduser("./taxonomy_mini_ready.csv")
df = pd.read_csv(path_csv)
primary_label = df["primary_label"].unique()

# Create label to target dict
n_classes = len(primary_label)
DICTIONARY_LABEL2TARGET = {}
for i, label in enumerate(primary_label):
    vec = np.zeros(n_classes) 
    vec[i] = 1
    DICTIONARY_LABEL2TARGET[label] = vec

# Create a dictionary label to sample weight (sampled by train audio only)
df_audio = pd.read_csv("./train_mini_ready.csv")
temp_dict = df_audio["primary_label"].value_counts().to_dict()
total_count = len(df_audio)

DICTIONARY_LABEL2WEIGHT = {}
for label, count in temp_dict.items():
    w = (count / total_count)
    DICTIONARY_LABEL2WEIGHT[label] = w

In [3]:
class UtilFunctions():
    '''
    Utility class for handling label preprocessing in multi-label
    audio classification tasks.

    Includes:
    - Multi-hot encoding generation
    - Label aggregation for soundscapes
    - Label smoothing for regularization
    '''

    def __init__(self):
        # Initialize dictionaries and number of classes
        self.label2target = DICTIONARY_LABEL2TARGET
        self.label2weight = DICTIONARY_LABEL2WEIGHT
        self.n_classes = len(self.label2target)

    def multi_hot_audios(self, primary_label, secondary_labels):
        # Start from primary label one-hot vector (copy to avoid mutation)
        vec_1 = self.label2target[primary_label].copy()

        # If no secondary labels, return primary vector
        if secondary_labels == "[]":
            return vec_1
        else:
            # Parse string representation of list into Python list
            s_labels = ast.literal_eval(secondary_labels)

            # Combine vectors via element-wise max (multi-hot encoding)
            for label in s_labels:
                if label in self.label2target:
                    vec_2 = self.label2target[label]
                    vec_1 = np.maximum(vec_1, vec_2)
            return vec_1
        
    def multi_hot_soundscapes(self, primary_label):
        # Split multiple labels separated by ';'
        p_labels = primary_label.split(";")

        # Initialize empty multi-hot vector
        vec_1 = np.zeros(self.n_classes)

        # Aggregate all label vectors into a single multi-hot vector
        for label in p_labels:
            vec_2 = self.label2target[label]
            vec_1 = np.maximum(vec_1, vec_2)
        return vec_1
    
    # LABEL SMOOTHING (For knowledge distilation)
    def label_smooth(self, y_true, mu = 0.05):
        y_true_smooth = y_true * (1 - mu)  +  (mu * torch.sum(y_true, dim=1, keepdim=True) / self.n_classes) 
        return y_true_smooth

# MODEL

In [4]:
class BirdEfficientNetV2S(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, dropout=0.25, n_fft=2048, hop_length=512, n_mels=128):

        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        
        # CNN backbone
        self.backbone = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k", 
                        pretrained=pretrained, in_chans=1, num_classes=0, global_pool="avg")

        # Audio to Log-Mel
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=32000, n_fft=self.n_fft, 
                    hop_length=self.hop_length, n_mels=self.n_mels, power=2.0, f_min=20, normalized=True)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)


        # Classification head
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        """
        Args = x: input tensor of shape [B, 32kHz * 5s]

        Returns =  y: raw class scores of shape [B, num_classes]
        """
        # Transform waveform to log-mel
        x = self.mel(x)  # [B,128,T]
        x = self.db(x)  
        
        # z Normalize
        global_mean = -19.631881713867188
        global_std = 18.53003692626953
        #mean = x.mean(dim=2, keepdim=True)  # media temporal por banda mel
        #std = x.std(dim=2, keepdim=True)    # desviación temporal por banda mel
        
        x = (x - global_mean) / (global_std + 1e-6)
        
        # CNN
        x = x.unsqueeze(1)  # [B,1,128,T]
        x = self.backbone(x)  # [B,feat_dim]
        # Head
        y = self.head(x)  # [B,234]

        return y

# TEST DATASET

In [5]:
class BirdSoundscapeDataset(Dataset):
    '''
    Inputs:
    - df_soundscapes ("train_soundscapes_labels.csv" dataframe with filename, start, end, target information)

    Outputs:
    - waveform: Tensor [samples]
    - target: Tensor [num_classes]

    Description:
    Loads audio chunks from soundscapes using timestamps and returns
    waveform with multi-hot labels.
    '''
    def __init__(self, df_soundscapes):
        self.df_soundscapes = df_soundscapes
        self.dir_soundscapes = os.path.expanduser("~/1_machine_learning_projects/BIRDS/data/train_soundscapes")
        self.utils = UtilFunctions()

    # Convert HH:MM:SS → frame index (samples)
    def time_to_sec(self, t):
        h, m, s = map(int, t.split(':'))
        return (h*3600 + m*60 + s) * 32000

    def __len__(self):
        return len(self.df_soundscapes)
    
    def __getitem__(self, idx):
        # Get rows
        row = self.df_soundscapes.iloc[idx]

        # Create path to load file 
        filename = row["filename"]
        path_filename = os.path.join(self.dir_soundscapes, filename)

        # Get time to load audio chunk
        start = self.time_to_sec(row["start"])
        end = self.time_to_sec(row["end"])

        # Load waveform 
        waveform, sample_rate =  sf.read(path_filename, start=start, stop=end)
        waveform = torch.tensor(waveform, dtype=torch.float32)
        
        # Get target from string
        target = self.utils.multi_hot_soundscapes(row["primary_label"])
        target = torch.tensor(target, dtype=torch.float32)

        return waveform, target
    

# TEST

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset
df = pd.read_csv("./train_soundscapes_labels_mini_ready.csv")
data_test = BirdSoundscapeDataset(df)

# Test DataLoader
loader_test = DataLoader(data_test, batch_size=32, shuffle=False)

# Load models


model_1 = BirdEfficientNetV2S(dropout=0.25, n_fft=2048, hop_length=512, n_mels=256).to(device)
model_1.load_state_dict(torch.load("./output/model_n_fft2048_hop_length512_n_mels256.pt", map_location=device))

model_2 = BirdEfficientNetV2S(dropout=0.25, n_fft=1024, hop_length=320, n_mels=128).to(device)
model_2.load_state_dict(torch.load("./output/model_n_fft1024_hop_length320_n_mels128_p_soundscape_noise50_no_norm.pt", map_location=device))

# Second best
model_3 = BirdEfficientNetV2S(dropout=0.25, n_fft=2048, hop_length=512, n_mels=256).to(device)
model_3.load_state_dict(torch.load("./output/model_n_fft2048_hop_length512_n_mels256_p_soundscape_noise50_ultimate_global_mean.pt", map_location=device))

# Best with 
model_4 = BirdEfficientNetV2S(dropout=0.25, n_fft=2048, hop_length=512, n_mels=256).to(device)
model_4.load_state_dict(torch.load("./output/model_n_fft2048_hop_length512_n_mels256_p_soundscape_noise50_ultimate.pt", map_location=device))


# Test
pred_1 = []
pred_2 = []
pred_3 = []
pred_4 = []
targets = []
smooth_y = UtilFunctions() 

model_1.eval()
model_2.eval()
model_3.eval()
model_4.eval()
with torch.no_grad():

    pbar = tqdm(loader_test)
    for batch_X, batch_y in pbar:

        # Move tensors to GPU/CPU
        batch_X = batch_X.to(device)

        # Forward pass
        y_pred1 = model_1(batch_X)
        y_pred2 = model_2(batch_X)
        y_pred3 = model_3(batch_X)
        y_pred4 = model_4(batch_X)
        
        # Convert logits -> probabilities
        y_pred1 = torch.sigmoid(y_pred1)
        y_pred2 = torch.sigmoid(y_pred2)
        y_pred3 = torch.sigmoid(y_pred3)
        y_pred4 = torch.sigmoid(y_pred4)
        
        # Store predictions and targets
        pred_1.append(y_pred1.cpu())
        pred_2.append(y_pred2.cpu())
        pred_3.append(y_pred3.cpu())
        pred_4.append(y_pred4.cpu())
        targets.append(batch_y.cpu())

# ROC-AUC macro
#score = roc_auc_score(all_targets.numpy(), all_preds.numpy(), average="macro")
#print("ROC-AUC:", score)

# Concatenar batches
pred_1 = torch.cat(pred_1, dim=0)
pred_2 = torch.cat(pred_2, dim=0)
pred_3 = torch.cat(pred_3, dim=0)
pred_4 = torch.cat(pred_4, dim=0)

y_true = torch.cat(targets, dim=0)

100%|███████████████████████████████████████████████████████████████████████████| 9/9 [00:21<00:00,  2.37s/it]


# SCORES

In [7]:
import numpy as np
import torch
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    jaccard_score,
    accuracy_score,
)

def evaluate_multilabel(y_prob, y_true, threshold=0.5):
    """
    y_prob: probabilidades sigmoid, shape [N, C]
    y_true: etiquetas multi-hot 0/1, shape [N, C]
    threshold: umbral para convertir probabilidades a 0/1
    """

    # Convierte torch.Tensor a numpy
    if isinstance(y_prob, torch.Tensor):
        y_prob = y_prob.detach().cpu().numpy()
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.detach().cpu().numpy()

    # Asegura arrays
    y_prob = np.asarray(y_prob)
    y_true = np.asarray(y_true).astype(int)

    # Predicción binaria final
    y_pred = (y_prob >= threshold).astype(int)

    scores = {}

    # Métricas con predicciones binarias
    scores["subset_accuracy"] = accuracy_score(y_true, y_pred)
    scores["hamming_loss"] = hamming_loss(y_true, y_pred)

    scores["precision_micro"] = precision_score(y_true, y_pred, average="micro", zero_division=0)
    scores["recall_micro"] = recall_score(y_true, y_pred, average="micro", zero_division=0)
    scores["f1_micro"] = f1_score(y_true, y_pred, average="micro", zero_division=0)

    scores["precision_macro"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
    scores["recall_macro"] = recall_score(y_true, y_pred, average="macro", zero_division=0)
    scores["f1_macro"] = f1_score(y_true, y_pred, average="macro", zero_division=0)

    scores["jaccard_samples"] = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

    valid = (y_true.sum(axis=0) > 0) & (y_true.sum(axis=0) < y_true.shape[0])

    scores["roc_auc_macro_birdclef"] = roc_auc_score(y_true[:, valid], y_prob[:, valid], average="macro")

    return scores, y_pred

In [8]:


# Evaluar cada modelo individualmente
scores_1, y_bin_1 = evaluate_multilabel(pred_1, y_true, threshold=0.5)
scores_2, y_bin_2 = evaluate_multilabel(pred_2, y_true, threshold=0.5)
scores_3, y_bin_3 = evaluate_multilabel(pred_3, y_true, threshold=0.5)
scores_4, y_bin_4 = evaluate_multilabel(pred_4, y_true, threshold=0.5)

# Imprimir comparación
for name, scores in {
    "model_1": scores_1,
    "model_2": scores_2,
    "model_3": scores_3,
    "model_4": scores_4,
}.items():
    print(f"\n{name}")
    for k, v in scores.items():
        print(f"{k}: {v:.5f}")


model_1
subset_accuracy: 0.33451
hamming_loss: 0.25528
precision_micro: 0.60246
recall_micro: 0.43235
f1_micro: 0.50342
precision_macro: 0.66143
recall_macro: 0.37621
f1_macro: 0.40075
jaccard_samples: 0.42254
roc_auc_macro_birdclef: 0.76494

model_2
subset_accuracy: 0.33803
hamming_loss: 0.28785
precision_micro: 0.52381
recall_micro: 0.42059
f1_micro: 0.46656
precision_macro: 0.44358
recall_macro: 0.40975
f1_macro: 0.38710
jaccard_samples: 0.41549
roc_auc_macro_birdclef: 0.63440

model_3
subset_accuracy: 0.40493
hamming_loss: 0.22711
precision_micro: 0.69340
recall_micro: 0.43235
f1_micro: 0.53261
precision_macro: 0.59841
recall_macro: 0.38021
f1_macro: 0.42322
jaccard_samples: 0.45599
roc_auc_macro_birdclef: 0.74020

model_4
subset_accuracy: 0.46479
hamming_loss: 0.18750
precision_micro: 0.71972
recall_micro: 0.61176
f1_micro: 0.66137
precision_macro: 0.74452
recall_macro: 0.58298
f1_macro: 0.60285
jaccard_samples: 0.58744
roc_auc_macro_birdclef: 0.85486


In [9]:
class predict():
    def __init__(self, filename, model, mode : str = "mean", n = 1):
        self.path_test = "./test_soundscapes/"
        self.filename = filename
        self.sample_rate = 32000
        self.mode = mode
        self.model = model
        self.n = n

    def get_audio_prediction(self):
        # Load waveform
        path_wave = os.path.join(self.path_test, self.filename)
        waveform, _ = sf.read(path_wave)
        segment_samples = int(5 * self.sample_rate)
        # Cut waveform in 12 chunks of 5s each and save name and prediction
        predictions = []
        filenames = []
        self.model.eval()
        for i in range(12):
            # audio to pred
            print(i)        
            start =  i * segment_samples
            end = start + segment_samples
            chunk = waveform[start:end]
            y_pred = self.model.predict(chunk)
            predictions.append(y_pred)

            # Filename
            end = "_" + str((i + 1) * 5)
            f_name = self.filename.replace(".ogg", "") + end
            filenames.append(f_name)
        return filenames, torch.stack(predictions)
    
    def mean_method(self, predictions):
        predictions_mean  = torch.mean(predictions, dim=0)
        new_pred = predictions * predictions_mean
        return new_pred

    def top_n_method(self, predictions):
        vals, _ = torch.topk(predictions, k=self.n, dim=0)
        vals_mean = torch.mean(vals, dim = 0)
        new_pred = predictions * vals_mean
        return new_pred

    def conv_method(self, predictions):
    # predictions: [12, 234]
        x = predictions.T.unsqueeze(0)
        kernel = torch.tensor([0.25, 0.5, 0.25], dtype=x.dtype, device=x.device).view(1, 1, 3)
        kernel = kernel.repeat(predictions.shape[1], 1, 1)
        out = F.conv1d(x, kernel, padding=1, groups=predictions.shape[1])
        return out.squeeze(0).T
        
    def get_final_prediction(self, n):
        # Get final predictions and filenames
        name2pred = {}
        file_names, predictions  = self.get_audio_prediction()
        if self.mode == "mean":
            new_predictions = self.mean_method(predictions)
        elif self.mode == "top_n":
            new_predictions = self.top_n_method(predictions)
        for name, pred in zip(file_names, new_predictions):
            name2pred[name] = pred
        return name2pred


t = pred_1
p = predict("v", 4)

print(t, "\nmean")

print(p.mean_method(t), "\ntopn")

print(p.top_n_method(t), "\nconv")

print(p.conv_method(t))

tensor([[0.3990, 0.3884, 0.3528, 0.3357],
        [0.3129, 0.5104, 0.2683, 0.3044],
        [0.4424, 0.4152, 0.2659, 0.3873],
        ...,
        [0.2547, 0.2700, 0.1901, 0.6462],
        [0.3136, 0.2288, 0.1449, 0.7480],
        [0.3518, 0.4144, 0.2496, 0.3680]]) 
mean
tensor([[0.1521, 0.1739, 0.0786, 0.1353],
        [0.1193, 0.2285, 0.0598, 0.1227],
        [0.1686, 0.1859, 0.0593, 0.1561],
        ...,
        [0.0971, 0.1208, 0.0424, 0.2605],
        [0.1195, 0.1024, 0.0323, 0.3016],
        [0.1341, 0.1855, 0.0556, 0.1484]]) 
topn
tensor([[0.3387, 0.3639, 0.1949, 0.2923],
        [0.2656, 0.4782, 0.1482, 0.2651],
        [0.3755, 0.3890, 0.1469, 0.3373],
        ...,
        [0.2162, 0.2529, 0.1051, 0.5628],
        [0.2662, 0.2144, 0.0801, 0.6514],
        [0.2986, 0.3882, 0.1379, 0.3205]]) 
conv
tensor([[0.2777, 0.3218, 0.2435, 0.2440],
        [0.3668, 0.4561, 0.2888, 0.3330],
        [0.3882, 0.4472, 0.2700, 0.3555],
        ...,
        [0.2627, 0.2851, 0.1593, 0.6938],
   

In [10]:
scores_normal, _ = evaluate_multilabel(pred_1, y_true, threshold=0.5)
scores_mean, _ = evaluate_multilabel(p.mean_method(pred_1), y_true, threshold=0.5)
scores_top, _ = evaluate_multilabel(p.top_n_method(pred_1), y_true, threshold=0.5)
scores_conv, _ = evaluate_multilabel(p.conv_method(pred_1), y_true, threshold=0.5)

# Imprimir comparación
for name, scores in {
    "model_1": scores_normal,
    "model_2": scores_mean,
    "model_3": scores_top,
    "model_4": scores_conv,
}.items():
    print(f"\n{name}")
    for k, v in scores.items():
        print(f"{k}: {v:.5f}")


model_1
subset_accuracy: 0.33451
hamming_loss: 0.25528
precision_micro: 0.60246
recall_micro: 0.43235
f1_micro: 0.50342
precision_macro: 0.66143
recall_macro: 0.37621
f1_macro: 0.40075
jaccard_samples: 0.42254
roc_auc_macro_birdclef: 0.76494

model_2
subset_accuracy: 0.00000
hamming_loss: 0.29930
precision_micro: 0.00000
recall_micro: 0.00000
f1_micro: 0.00000
precision_macro: 0.00000
recall_macro: 0.00000
f1_macro: 0.00000
jaccard_samples: 0.00000
roc_auc_macro_birdclef: 0.76494

model_3
subset_accuracy: 0.31338
hamming_loss: 0.25000
precision_micro: 0.64894
recall_micro: 0.35882
f1_micro: 0.46212
precision_macro: 0.45000
recall_macro: 0.30813
f1_macro: 0.35083
jaccard_samples: 0.37148
roc_auc_macro_birdclef: 0.76494

model_4
subset_accuracy: 0.37676
hamming_loss: 0.24648
precision_micro: 0.61905
recall_micro: 0.45882
f1_micro: 0.52703
precision_macro: 0.41771
recall_macro: 0.39168
f1_macro: 0.39647
jaccard_samples: 0.45951
roc_auc_macro_birdclef: 0.76770
